# Завдання 2. Багатошаровий перцептрон (MLP) для MNIST
Свої зображення покладіть у папку `my_digits/` (файли `0.png`, `1.png`, …, `9.png`). Цифра — темна на білому тлі.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# перші 4 зображення
plt.figure(figsize=(8,2.5))
for i in range(4):
    plt.subplot(1,4,i+1); plt.imshow(X_train[i], cmap='gray'); plt.title(f"Мітка: {y_train[i]}"); plt.axis('off')
plt.show()

In [ ]:
num_pixels = X_train.shape[1] * X_train.shape[2]          # 784
X_train = X_train.reshape(-1, num_pixels).astype('float32') / 255
X_test  = X_test.reshape(-1, num_pixels).astype('float32') / 255
y_train_c = to_categorical(y_train)
y_test_c  = to_categorical(y_test)
num_classes = y_test_c.shape[1]
print(X_train.shape, y_train_c.shape)

In [ ]:
def baseline_model():
    model = keras.Sequential([
        layers.Input(shape=(num_pixels,)),
        layers.Dense(num_pixels, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

model = baseline_model()
history = model.fit(X_train, y_train_c, validation_data=(X_test, y_test_c),
                    epochs=100, batch_size=200, verbose=2)   # за умовою: 100 епох, батч 200
scores = model.evaluate(X_test, y_test_c, verbose=0)
print("Помилка класифікації: %.2f%%" % (100 - scores[1]*100))

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(history.history['accuracy'], label='train'); ax[0].plot(history.history['val_accuracy'], label='val')
ax[0].set_title('Точність'); ax[0].legend()
ax[1].plot(history.history['loss'], label='train'); ax[1].plot(history.history['val_loss'], label='val')
ax[1].set_title('Втрати'); ax[1].legend()
plt.show()

**Покращення (спробуйте):** `Dropout(0.2)` після прихованого шару, `EarlyStopping(patience=5, restore_best_weights=True)`, менше епох (≈10–20), більше нейронів. Якщо val_loss росте, а train_loss падає — це перенавчання.

In [ ]:
model2 = keras.Sequential([
    layers.Input(shape=(num_pixels,)),
    layers.Dense(num_pixels, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')
])
model2.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
h2 = model2.fit(X_train, y_train_c, validation_data=(X_test, y_test_c),
                epochs=100, batch_size=200, callbacks=[es], verbose=2)
print("Покращена MLP, помилка: %.2f%%" % (100 - model2.evaluate(X_test, y_test_c, verbose=0)[1]*100))
model2.save('mlp_mnist.keras')     # збереження моделі

In [ ]:
# Розпізнавання прикладів з датасету
idx = list(range(10))
pred = model2.predict(X_test[idx], verbose=0).argmax(axis=1)
plt.figure(figsize=(12,2.5))
for k,i in enumerate(idx):
    plt.subplot(1,10,k+1); plt.imshow(X_test[i].reshape(28,28), cmap='gray'); plt.axis('off')
    plt.title(f"{pred[k]} ({'✓' if pred[k]==y_test[i] else '✗'})")
plt.show()

In [ ]:
# Розпізнавання власних зображень (MLP)
from PIL import Image, ImageOps
import os, glob

def load_my_image(path):
    img = Image.open(path).convert('L').resize((28,28))
    arr = 255 - np.array(img)              # інверсія: у MNIST біла цифра на чорному
    return arr.astype('float32') / 255

files = sorted(glob.glob('my_digits/*.png'))
plt.figure(figsize=(12,2.5))
for k,f in enumerate(files):
    a = load_my_image(f)
    p = model2.predict(a.reshape(1,-1), verbose=0)
    plt.subplot(1,len(files),k+1); plt.imshow(a, cmap='gray'); plt.axis('off')
    plt.title(f"файл {os.path.basename(f)[0]}\nпрогноз {p.argmax()}")
plt.show()